# Entrenamiento de de dataset sintetico MLP normalizado (0-1)
## SafeDrive — Pipeline completo

Ejecuta las secciones **en orden**:
1. Genera dataset sintético a escala real
2. Monta el dataset final, construir archivos train, test, validation
3. Entrena y exporta `.keras` + `.tflite`

In [ ]:
# Dependencias comunes a todo el notebook
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

print(f'TensorFlow {tf.__version__}')

---
## 1. Genera dataset sintético
Genera datos físicamente realistas para las 3 clases.

**Salida:** `ml/data/processed/source/datos_sinteticos.csv`

In [ ]:
for folder in ['ml/data/processed/source', 'ml/data/processed/dataset/wrong_data', 'models']:
    os.makedirs(folder, exist_ok=True)

def generar_datos(n_samples, label):
    if label == 0:  # Conducción Normal
        peak_g = np.random.uniform(0.8, 1.2, n_samples)      # 1G es reposo
        jerk = np.random.uniform(0.1, 2.0, n_samples)        # Cambios suaves
        audio = np.random.uniform(30.0, 60.0, n_samples)     # Ruido ambiente (dB)
        angular = np.random.uniform(0.0, 30.0, n_samples)    # Giro leve (º/s)
        velocidad = np.random.uniform(0, 120, n_samples)     # km/h
    elif label == 1:  # Conducción Agresiva / Frenazo
        peak_g = np.random.uniform(1.2, 4.0, n_samples)      # Cerca del umbral de alarma
        jerk = np.random.uniform(2.0, 7.0, n_samples)        # Movimientos bruscos
        audio = np.random.uniform(60.0, 85.0, n_samples)     # Frenazos/revoluciones
        angular = np.random.uniform(30.0, 120.0, n_samples)  # Volantazos bruscos
        velocidad = np.random.uniform(20, 140, n_samples)
    elif label == 2:  # ACCIDENTE (Impacto detectado)
        peak_g = np.random.uniform(4.0, 15.0, n_samples)     # Disparo de EDR (>4G)
        jerk = np.random.uniform(7.0, 20.0, n_samples)       # Impacto violento
        audio = np.random.uniform(90.0, 120.0, n_samples)    # Estruendo de choque
        angular = np.random.uniform(120.0, 500.0, n_samples) # Vuelcos o trompos
        velocidad = np.random.uniform(30, 200, n_samples)

    return pd.DataFrame({
        'Peak_G':         peak_g,
        'Jerk':           jerk,
        'Firma_Acustica': audio,
        'Cambio_Angular': angular,
        'Velocidad':      velocidad,
        'Label':          [label] * n_samples
    })

df_sintetico = pd.concat([
    generar_datos(4000, 0),
    generar_datos(1500, 1),
    generar_datos(1500, 2)
]).sample(frac=1, random_state=42).reset_index(drop=True)

# Dataset basura (outliers físicamente imposibles)
wrong_data = pd.DataFrame({
    'Peak_G':         [500.0, -2.0,   0.0,  999.9],
    'Jerk':           [0.0,  900.0, -10.0,    0.0],
    'Firma_Acustica': [200.0, -50.0, 0.0, 1000.0],
    'Cambio_Angular': [1000.0, -100.0, 0.0, 5000.0],
    'Velocidad':      [-50,    400,     0,      0],
    'Label':          [0,       1,     9,      2]
})

df_sintetico.to_csv('ml/data/processed/source/datos_sinteticos.csv', index=False)
wrong_data.to_csv('ml/data/processed/dataset/wrong_data/corrupted_data.csv', index=False)

print(f'Sintético: {len(df_sintetico)} filas')
print(df_sintetico['Label'].value_counts().sort_index())

---
## 2. Monta el dataset final
Fusiona todas las fuentes disponibles en `ml/data/processed/source/`, mezcla y divide en **train 70% / validation 15% / test 15%**.

In [ ]:
for folder in ['ml/data/processed/dataset/train',
               'ml/data/processed/dataset/validation',
               'ml/data/processed/dataset/test']:
    os.makedirs(folder, exist_ok=True)

# Leer todas las fuentes (wrong_data queda excluido por estar en otra carpeta)
archivos = glob.glob('ml/data/processed/source/*.csv')
print(f'Fuentes encontradas ({len(archivos)}):')
for f in archivos:
    df_tmp = pd.read_csv(f)
    print(f'  {f}  ->  {len(df_tmp)} filas')

dataset_completo = pd.concat(
    [pd.read_csv(f) for f in archivos], ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nTotal fusionado: {len(dataset_completo)} filas')
print(f'Distribución de clases:\n{dataset_completo["Label"].value_counts().sort_index()}')

# Split 70 / 15 / 15
df_train, df_temp = train_test_split(dataset_completo, test_size=0.30, random_state=42)
df_val,   df_test = train_test_split(df_temp,          test_size=0.50, random_state=42)

df_train.to_csv('ml/data/processed/dataset/train/train.csv',         index=False)
df_val.to_csv('ml/data/processed/dataset/validation/validation.csv', index=False)
df_test.to_csv('ml/data/processed/dataset/test/test.csv',            index=False)

print(f'\nTrain: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

---
## 3. Entrena y exporta modelos
Entrena la red neuronal y exporta dos formatos:
- **`.keras`** — para reentrenar o inspeccionar en Python
- **`.tflite`** — para desplegar en el dispositivo Android

In [ ]:
# 1. Cargar los splits generados
df_train = pd.read_csv('ml/data/processed/dataset/train/train.csv')
df_val   = pd.read_csv('ml/data/processed/dataset/validation/validation.csv')
df_test  = pd.read_csv('ml/data/processed/dataset/test/test.csv')

# 2. Separar Features (X) y Labels (y)
X_train = df_train.drop(columns=['Label']).values;  y_train = df_train['Label'].values
X_val   = df_val.drop(columns=['Label']).values;    y_val   = df_val['Label'].values
X_test  = df_test.drop(columns=['Label']).values;   y_test  = df_test['Label'].values

n_features = X_train.shape[1]
print(f'Features: {n_features} | Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

In [ ]:
# Esta capa aprenderá la media y varianza de X_train y se guardará dentro del .tflite
normalizer = keras.layers.Normalization(axis=-1)
normalizer.adapt(X_train) 

# 3. Definir modelo con el Normalizer incluido como primera capa
model = keras.Sequential([
    keras.layers.InputLayer(input_shape=(n_features,)),
    normalizer, # <- El modelo escala los datos automáticamente aquí
    keras.layers.Dense(24, activation='relu'),
    keras.layers.Dense(12, activation='relu'),
    keras.layers.Dense(3,  activation='softmax')
], name='safedrive_cu03_v2')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# 4. Entrenamiento
print("\nIniciando entrenamiento...")
history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=16,
    validation_data=(X_val, y_val),
    verbose=1
)

In [ ]:
# 5. Evaluación sobre test
print("\n--- Resultados Finales ---")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss:     {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

clases = ['Falso Positivo', 'Posible Susto', 'Choque Grave']
predicciones = model.predict(X_test, verbose=0)

print('\nPrimeras 10 predicciones:')
for i in range(min(10, len(y_test))):
    idx  = np.argmax(predicciones[i])
    real = int(y_test[i])
    marca = 'OK' if idx == real else 'ERROR'
    print(f'  [{marca}]  Pred: {clases[idx]:20s} | Real: {clases[real]}')

In [ ]:
# 6. Exportación (Keras y TFLite)
os.makedirs('models', exist_ok=True)

# Formato Keras
keras_path = 'models/safedrive_cu03_model_v2.keras'
model.save(keras_path)
print(f'\nKeras  guardado en: {keras_path}  ({os.path.getsize(keras_path)/1024:.1f} KB)')

# Formato TFLite (Con optimización de tamaño y batería para Android)
tflite_path = 'models/safedrive_cu03_model_v2.tflite'
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] # <- Cuantización activada (Mejora 4)
tflite_model = converter.convert()

with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f'TFLite guardado en: {tflite_path}  ({os.path.getsize(tflite_path)/1024:.1f} KB)')
print("\n¡Listo! El modelo TFLite ya incluye el escalado de datos internamente.")